# CredResolve — Source Profiling & Identity Audit

This notebook profiles the raw extracts and documents entity-key reliability. It does not alter raw data or infer identity where mappings are ambiguous.

In [1]:
from pathlib import Path
import sqlite3, pandas as pd, numpy as np
candidates=[Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]
ROOT=next((p.resolve() for p in candidates if (p/'data/processed/collections.db').exists()),Path.cwd().resolve())
con=sqlite3.connect(ROOT/'data/processed/collections.db')


## 1. Dataset inventory

In [2]:
tables=pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' AND name LIKE 'raw_%' ORDER BY name",con)
rows=[]
for name in tables.name:
 d=pd.read_sql_query(f'SELECT * FROM "{name}"',con)
 rows.append({'dataset':name,'rows':len(d),'columns':len(d.columns),'missing_cells':int(d.isna().sum().sum()),'missing_pct':round(d.isna().sum().sum()/max(1,d.size)*100,2)})
inventory=pd.DataFrame(rows).sort_values('rows',ascending=False)
display(inventory)


,dataset,rows,columns,missing_cells,missing_pct
5,raw_call_attempts,120000,9,2400,0.22
7,raw_calls,91350,11,1827,0.18
16,raw_whatsapp_events,60600,8,0,0.00
0,raw_account_status_history,60000,8,0,0.00
10,raw_daily_targeting,45000,7,0,0.00
14,raw_sms_events,45000,8,0,0.00
6,raw_call_dispositions,35000,8,0,0.00
4,raw_borrowers,30600,8,1509,0.62
1,raw_accounts,30000,11,455,0.14
3,raw_agents,30000,8,0,0.00


## 2. Key uniqueness and cross-account collisions

In [3]:
checks=[('raw_accounts','account_id'),('raw_agents','agent_id'),('raw_payments','payment_id'),('raw_call_attempts','attempt_id'),('raw_calls','call_id'),('raw_call_dispositions','disposition_id'),('raw_field_visits','visit_id'),('raw_promises_to_pay','ptp_id'),('raw_daily_targeting','target_id')]
key_rows=[]
for table,key in checks:
 d=pd.read_sql_query(f'SELECT "{key}" FROM "{table}"',con)
 key_rows.append({'table':table,'key':key,'rows':len(d),'distinct_non_null':d[key].nunique(),'null_keys':int(d[key].isna().sum()),'duplicate_excess':int(d[key].notna().sum()-d[key].nunique())})
display(pd.DataFrame(key_rows))


,table,key,rows,distinct_non_null,null_keys,duplicate_excess
0,raw_accounts,account_id,30000,30000,0,0
1,raw_agents,agent_id,30000,1000,0,29000
2,raw_payments,payment_id,25500,25000,0,500
3,raw_call_attempts,attempt_id,120000,120000,0,0
4,raw_calls,call_id,91350,90000,0,1350
5,raw_call_dispositions,disposition_id,35000,35000,0,0
6,raw_field_visits,visit_id,25000,25000,0,0
7,raw_promises_to_pay,ptp_id,18000,18000,0,0
8,raw_daily_targeting,target_id,45000,45000,0,0


## 3. Account–borrower cross-source audit

In [4]:
acc=pd.read_sql_query('SELECT account_id, borrower_id FROM raw_accounts',con)
identity=[]
for table in ['raw_payments','raw_calls','raw_call_attempts','raw_call_dispositions','raw_field_visits','raw_promises_to_pay']:
 d=pd.read_sql_query(f'SELECT account_id, borrower_id FROM "{table}"',con)
 j=d.merge(acc,on='account_id',how='left',suffixes=('_event','_master'))
 mismatch=(j.borrower_id_event.notna() & j.borrower_id_master.notna() & j.borrower_id_event.ne(j.borrower_id_master))
 identity.append({'source':table,'rows':len(j),'mismatch_rows':int(mismatch.sum()),'missing_master_borrower':int(j.borrower_id_master.isna().sum())})
display(pd.DataFrame(identity))


,source,rows,mismatch_rows,missing_master_borrower
0,raw_payments,25500,25113,386
1,raw_calls,91350,89939,1403
2,raw_call_attempts,120000,118126,1863
3,raw_call_dispositions,35000,34474,524
4,raw_field_visits,25000,24621,377
5,raw_promises_to_pay,18000,17706,293


## 4. Agent identifier ambiguity

In [5]:
agents=pd.read_sql_query('SELECT * FROM raw_agents',con)
agent_summary=pd.DataFrame([
 {'test':'agent_id count','value':agents.agent_id.nunique()},
 {'test':'employee_code count','value':agents.employee_code.nunique()},
 {'test':'agent IDs with >1 employee code','value':int((agents.groupby('agent_id').employee_code.nunique()>1).sum())},
 {'test':'employee codes with >1 agent ID','value':int((agents.groupby('employee_code').agent_id.nunique()>1).sum())},
 {'test':'agent IDs with >1 vendor','value':int((agents.groupby('agent_id').vendor_id.nunique()>1).sum())}
])
display(agent_summary)


,test,value
0,agent_id count,1000
1,employee_code count,1099
2,agent IDs with >1 employee code,1000
3,employee codes with >1 agent ID,1099
4,agent IDs with >1 vendor,1000


## 5. Interpretation and controls

- Use `account_id` as the account-level analytical key; preserve source borrower IDs and master borrower IDs separately.
- Do not force-resolve conflicting borrower or agent identities.
- Do not join calls, attempts and dispositions on `call_id` alone: account-level consistency must be validated first.
- Do not use `payment_reference` as a transaction key; use `payment_id` after processor confirmation and quarantine conflicting groups.
- Source profiling identifies quality issues; it does not establish operational causality.

In [6]:
con.close()
print("Profiling complete; raw data unchanged.")

Profiling complete; raw data unchanged.
